
### For testing 

In [2]:
import nidaqmx
from nidaqmx.constants import ThermocoupleType, TemperatureUnits, ResistanceConfiguration, RTDType, ResistanceUnits, BridgeConfiguration, ExcitationSource, CurrentUnits
import matplotlib
import time

In [ ]:
# listing all the devices 

system = nidaqmx.system.System.local()

print("--- Connected NI Devices ---")
for device in system.devices:
    print(f"Device Name: {device.name} | Model: {device.product_type}")

--- Connected NI Devices ---
Device Name: cDAQ1 | Model: cDAQ-9171
Device Name: cDAQ1Mod1 | Model: NI 9219
Device Name: myDAQ1 | Model: NI myDAQ


In [ ]:
# reading Temperature
with nidaqmx.Task() as MeasureTemperature_Task :
    
    MeasureTemperature_Task.ai_channels.add_ai_thrmcpl_chan( # create temperature measure channel
        physical_channel = "cDAQ1Mod1/ai0",
        max_val = 100.0,
        min_val = -100.0,
        name_to_assign_to_channel = "Temperature Measuring",
        thermocouple_type = ThermocoupleType.K,
        units = TemperatureUnits.DEG_C,
    )

    print ("Temperature Measurement Started")

    try:
        while True:
            temperature = MeasureTemperature_Task.read() # read temperature
            print(f"Temperature: {temperature:.2f} °C")
            time.sleep(.5)

    except KeyboardInterrupt:
        print("Temperature Measurement Stopped")



Temperature Measurement Started
Temperature: 25.84 °C
Temperature: 25.83 °C
Temperature: 25.83 °C
Temperature Measurement Stopped


In [13]:
# measure voltage

with nidaqmx.Task() as MeasureVoltage_Task :

    MeasureVoltage_Task.ai_channels.add_ai_voltage_chan(
        physical_channel = "cDAQ1Mod1/ai1",
        max_val= 60.0,
        min_val = 0.0,
        name_to_assign_to_channel = "Voltage Measuring"
    )

    print ("Measuring voltage")

    try :

        while True:
            vol = MeasureVoltage_Task.read() # read voltage
            print(f"Voltage: {vol:.6f} V")

    except KeyboardInterrupt:
        print("Voltage Measurement Stopped")


Measuring voltage
Voltage: -0.000100 V
Voltage: -0.000079 V
Voltage: -0.000050 V
Voltage: -0.000079 V
Voltage: -0.000064 V
Voltage: -0.000072 V
Voltage: -0.000122 V
Voltage: -0.000043 V
Voltage: -0.000043 V
Voltage: -0.000086 V
Voltage: 0.000036 V
Voltage: -0.000007 V
Voltage: -0.000072 V
Voltage: 0.000014 V
Voltage Measurement Stopped


In [15]:
# measure Current
with nidaqmx.Task() as MeasureCurrent_Task:

    MeasureCurrent_Task.ai_channels.add_ai_current_chan(
        physical_channel = "cDAQ1Mod1/ai1",
        max_val = 0.02,
        min_val = -0.02,
        name_to_assign_to_channel = "Current Measuring"
    )
    print ("Measuring Current")

    try:
        while True:
            current = MeasureCurrent_Task.read() # read current
            print(f"Current: {current*1000:.6f} mA")

    except KeyboardInterrupt:
        print("Current Measurement Stopped")    
        

Measuring Current
Current: 0.002220 mA
Current: 0.002205 mA
Current: 0.002229 mA
Current: 0.002193 mA
Current: 0.002185 mA
Current: 0.002185 mA
Current Measurement Stopped


In [8]:
# measure resistance - 2 wire

with nidaqmx.Task() as res_task:

    res_task.ai_channels.add_ai_resistance_chan(
        physical_channel="cDAQ1Mod1/ai1",
        name_to_assign_to_channel="TwoWireResistor Measurement",
        min_val=0.0,
        max_val=10000.0,  
        resistance_config=ResistanceConfiguration.TWO_WIRE,  
        current_excit_source=ExcitationSource.INTERNAL,     # ni 9219 source is internal     
        current_excit_val=500e-6                            # driver value               
    )
    
    print("Reading 2-Wire Resistance")

    try:
        while True:
            ohms = res_task.read()
            print(f"Measured Resistance: {ohms:.6f} Ohms")
            time.sleep(0.5)

    except KeyboardInterrupt:
        print("\nMeasurement stopped safely.")

Reading 2-Wire Resistance
Measured Resistance: 11.985631 Ohms
Measured Resistance: 11.986882 Ohms
Measured Resistance: 12.116433 Ohms

Measurement stopped safely.


In [10]:
# measure Resistance - 4 wire
with nidaqmx.Task() as MeasureResistance_4wire_Task:

    MeasureResistance_4wire_Task.ai_channels.add_ai_resistance_chan(
        physical_channel="cDAQ1Mod1/ai1",
        name_to_assign_to_channel="FourWireResistor Measurement",
        min_val=0.0,
        max_val=10000.0,
        current_excit_source=ExcitationSource.INTERNAL,
        current_excit_val=500e-6,
        resistance_config=ResistanceConfiguration.FOUR_WIRE,
        units=ResistanceUnits.OHMS

    )


    print ("Measuring 4-Wire Resistance")

    try: 
        while True:

            resis = MeasureResistance_4wire_Task.read()
            print (f"Resistance: {resis:.6f} Ohms")

    except KeyboardInterrupt:
        print("4-Wire Resistance Measurement Stopped")

Measuring 4-Wire Resistance
Resistance: 0.000000 Ohms
Resistance: 0.001252 Ohms
Resistance: 0.003129 Ohms
Resistance: 0.008136 Ohms
Resistance: 0.008136 Ohms
Resistance: 0.006258 Ohms
Resistance: 0.013143 Ohms
Resistance: 0.015646 Ohms
Resistance: 0.017524 Ohms
Resistance: 0.019401 Ohms
4-Wire Resistance Measurement Stopped


In [16]:
# for Resistance Temperature Measurement (RTD) -  2 wire

import time
import nidaqmx
from nidaqmx.constants import ResistanceConfiguration, ExcitationSource

def ohms_to_celsius_pt100(ohms):
    return (ohms - 100.0) / 0.385

with nidaqmx.Task() as res_task:
    # 1. Read the sensor as a raw 2-Wire Resistance channel (which IS supported)
    res_task.ai_channels.add_ai_resistance_chan(
        physical_channel="cDAQ1Mod1/ai0",
        name_to_assign_to_channel="TwoWireRTD_As_Ohms",
        min_val=0.0,
        max_val=10000.0,
        resistance_config=ResistanceConfiguration.TWO_WIRE, # This works here!
        current_excit_source=ExcitationSource.INTERNAL,
        current_excit_val=500e-6   
        )
    
    print("Reading 2-Wire RTD via Resistance Workaround")
    try:
        while True:
            raw_ohms = res_task.read()
            
            temperature = ohms_to_celsius_pt100(raw_ohms)
            
            print(f"Resistance: {raw_ohms:.2f} Ω | Calculated Temp: {temperature:.2f} °C")
            time.sleep(0.5)
            
    except KeyboardInterrupt:
        print("\nMeasurement stopped safely.")

Reading 2-Wire RTD via Resistance Workaround
Resistance: 10500.00 Ω | Calculated Temp: 27012.99 °C
Resistance: 10500.00 Ω | Calculated Temp: 27012.99 °C

Measurement stopped safely.


In [15]:
# measure RTD - 3 wire

with nidaqmx.Task() as MeasureRTD_3wire_Task:

    MeasureRTD_3wire_Task.ai_channels.add_ai_rtd_chan(
        physical_channel= "cDAQ1Mod1/ai0",
        name_to_assign_to_channel= "3-Wire RTD Measurement",
        resistance_config = ResistanceConfiguration.THREE_WIRE,
        current_excit_source= ExcitationSource.INTERNAL,
        current_excit_val= 500e-6,
        units = TemperatureUnits.DEG_C
    )

    print ("Measuring 3 wire RTD")


    try: 
        while True:
            res = MeasureRTD_3wire_Task.read()
            print(res)

    except KeyboardInterrupt:
        print ("Measuring Stopped")

Measuring 3 wire RTD
1351.669265513736
1351.669265513736
1351.669265513736
1351.669265513736
Measuring Stopped


In [14]:
# measure RTD - 4 wire

with nidaqmx.Task() as MeasureRTD_4wire_Task:
    MeasureRTD_4wire_Task.ai_channels.add_ai_rtd_chan(
        physical_channel= "cDAQ1Mod1/ai0",
        name_to_assign_to_channel= "4-Wire RTD Measurement",
        rtd_type= RTDType.PT_3750,
        resistance_config=ResistanceConfiguration.FOUR_WIRE,
        current_excit_source=ExcitationSource.INTERNAL,
        units = TemperatureUnits.DEG_C,
        current_excit_val=500e-6
    )

    print ("Measuring 4 wire RTD")

    try :
        i = 0
        while True:    
            res=MeasureRTD_4wire_Task.read()
            print(f"{i}. Measured RTD: {res:.5f}")
            i+=1

    except KeyboardInterrupt:
        print("Measuring stopped")

Measuring 4 wire RTD
0. Measured RTD: 1351.66927
1. Measured RTD: 1351.66927
2. Measured RTD: 1351.66927
3. Measured RTD: 1351.66927
Measuring stopped


In [ ]:
# bridge measurement

with nidaqmx.Task() as bridge_task:

    bridge_task.ai_channels.add_ai_bridge_chan(
        physical_channel="cDAQ1Mod1/ai0",
        name_to_assign_to_channel="LoadCell",
        min_val=-0.025, 
        max_val=0.025,
        bridge_config= BridgeConfiguration.FULL_BRIDGE
    )
    
    print("Reading bridge sensor data...")
    try:
        while True:
            raw_bridge = bridge_task.read()
            print(f"Bridge Ratio: {raw_bridge * 1000:.4f} mV/V")
            time.sleep(0.5)
    except KeyboardInterrupt:
        pass

Reading bridge sensor data...
Bridge Ratio: 0.0357 mV/V
Bridge Ratio: 0.0366 mV/V
Bridge Ratio: 0.0366 mV/V
Bridge Ratio: 0.0367 mV/V
Bridge Ratio: 0.0366 mV/V
Bridge Ratio: 0.0369 mV/V
Bridge Ratio: 0.0370 mV/V
Bridge Ratio: 0.0368 mV/V
Bridge Ratio: 0.0364 mV/V
Bridge Ratio: 0.0370 mV/V
Bridge Ratio: 0.0369 mV/V
Bridge Ratio: 0.0368 mV/V
Bridge Ratio: 0.0370 mV/V
Bridge Ratio: 0.0370 mV/V
Bridge Ratio: 0.0375 mV/V
Bridge Ratio: 0.0370 mV/V
Bridge Ratio: 0.2358 mV/V
Bridge Ratio: -0.0004 mV/V
Bridge Ratio: -0.0003 mV/V
Bridge Ratio: -0.0003 mV/V
Bridge Ratio: -0.0004 mV/V
Bridge Ratio: -0.0003 mV/V
Bridge Ratio: 0.0942 mV/V
Bridge Ratio: 0.0073 mV/V
Bridge Ratio: 0.0140 mV/V
Bridge Ratio: 0.0198 mV/V
Bridge Ratio: 0.0206 mV/V
Bridge Ratio: 0.0121 mV/V
Bridge Ratio: 0.0135 mV/V
Bridge Ratio: 0.0228 mV/V
Bridge Ratio: 0.0198 mV/V
Bridge Ratio: 0.0202 mV/V


In [41]:
# trying multi-channel

with nidaqmx.Task() as MultiChannel_Task:
    
    i = 4 # variable for setting multiple channels
    
    MultiChannel_Task.ai_channels.add_ai_thrmcpl_chan(
        physical_channel= f"cDAQ1Mod1/ai0:{i-1}",
        min_val= 0.0,
        max_val= 100.0,
        name_to_assign_to_channel="Multichannel Temperature",
        units= TemperatureUnits.DEG_F
        
    )
    
    print (f"Reading from {i} Channels")
    
    samples = MultiChannel_Task.read(number_of_samples_per_channel=5)

    for j in range(i):
        print(f"Channel {j} Samples: {samples[j]}")


Reading from 4 Channels
Channel 0 Samples: [25.491665538932857, 25.5007320645053, 25.509798417568813, 25.513531571601835, 25.52686402551118]
Channel 1 Samples: [23.892309904662774, 23.92333951940771, 23.912639852454873, 23.913709828638666, 23.922804541069084]
Channel 2 Samples: [33.771414770505274, 33.766113534942676, 33.770354524662636, 33.75710139805581, 33.7756557475265]
Channel 3 Samples: [-7212664347995.654, -7212664347995.654, -7212664347995.654, -7212664347995.654, -7212664347995.654]


In [ ]:
# creating a schema-ish thing so we can mapp out which channel has which task

Tasks = {
    "ai0" : "Temp",
    "ai1" : "Vol",
    "ai2" : "Curr",
    "ai3" : "Temp"

}

number_of_samples_per_channel = 4


Temp: list = []
Volt: list = []
Curr: list = []
Res: list = []





In [ ]:
with nidaqmx.Task() as Temp_Task:
    Temp_Task.ai_channels.add_ai_thrmcpl_chan(
        physical_channel=Temp,
        thermocouple_type= ThermocoupleType.K
    )
    
    Temp_Task.read(number_of_samples_per_channel= number_of_samples_per_channel)
    
    
with nidaqmx.Task() as Volt_Task:
    Volt_Task.ai_channels.add_ai_voltage_chan(
        physical_channel= Volt
    )
    Volt_Task.read(number_of_samples_per_channel= number_of_samples_per_channel)
    
    
with nidaqmx.Task() as Curr_Task:
    Curr_Task.ai_channels.add_ai_current_chan(
        physical_channel= Curr
        units= CurrentUnits.AMPS
    )
    Curr_Task.read(number_of_samples_per_channel= number_of_samples_per_channel)
    
    
with nidaqmx.Task() as Res_Task:
    Res_Task.ai_channels.add_ai_resistance_chan(
        physical_channel=Res,
        resistance_config= ResistanceConfiguration.TWO_WIRE
    )
    Res_Task.read(number_of_samples_per_channel= number_of_samples_per_channel)
    

